"""
OOP design part 1 - Car racing
Author: Mattias Westermark

Note: Gemini has been used as a sounding board
in accordance with the allowed aids policy.

Reference sites:
https://www.datacamp.com/tutorial/introduction-q-learning-beginner-tutorial
https://www.geeksforgeeks.org/machine-learning/q-learning-in-python/
"""

In [252]:
import numpy as np
import pandas as pd
import random

Constants for path and name of file containing race track in txt-format.

In [253]:
FILEPATH = "/workspaces/LTU-Programming-Portfolio/d0045e_ht2026/race_track/"
FILENAME = "auto2.txt"      # Edit filename to select track

Set hyperparameters (geeksforgeeks)

In [254]:
learning_rate = 0.8         # How much new info overrides old info.
discount_factor = 0.95      # How much future rewards are valued.
exploration_prob = 0.2      # Probability of taking a random action.
epochs = 10000              # Number of training episodes.

ReadTrackFile is used to read a file and store it in a list.

In [255]:
class ReadTrackFile:
    def __init__(self, fp, fn):
        self.filepath = fp
        self.filename = fn
        self.fullname = f"{self.filepath}{self.filename}"


    def load_track(self):
        with open(self.fullname, 'r', encoding='utf-8') as file:
            self.track_data = [line.strip('\n') for line in file.readlines()]
        return self.track_data

RaceTrack calls ReadTrackFile and maps the track.

In [256]:
class RaceTrack:
    def __init__(self):
        race_track = ReadTrackFile(FILEPATH, FILENAME)
        self.race_track_map = race_track.load_track()

        self.height = len(self.race_track_map)
        self.width = len(self.race_track_map[0])
        self.start = (0, 0)
        self.finish = (self.width - 1, self.height - 1)

        self.walls = set()
        self._parse_track()


    def _parse_track(self):                             # Identify the walls (non-empty positions) in the track file.
        for y, row in enumerate(self.race_track_map):
            for x, char in enumerate(row):
                if char != ' ':
                    self.walls.add((x, y))

        print(f"Walls are located at {self.walls}\n")


    def step(self, current_state, action):              # Geeksforgeeks
        y, x = divmod(current_state, self.width)

        new_x, new_y = x, y

        if action == 0:   # and x > 0:                                           # Left
            new_x -= 1
        elif action == 1: # and x < self.width - 1:                            # Right
            new_x += 1
        elif action == 2: # and y > 0:                                         # Up
            new_y -= 1
        elif action == 3: # and y < self.height - 1:                           # Down
            new_y += 1
        elif action == 4: # and x > 0 and y > 0:                               # Left and up
            new_x -= 1
            new_y -= 1
        elif action == 5: # and x < self.width - 1 and y < self.height -1:     # Right and down
            new_x += 1
            new_y += 1
        elif action == 6: # and x > 0 and y < self.height - 1:                 # Left and down
            new_x -= 1
            new_y += 1
        elif action == 7: # and x < self.width - 1 and y > 0:                  # Right and up
            new_x += 1
            new_y -= 1      

        done = False
        reward = -1

        if new_x < 0 or new_x >= self.width or new_y < 0 or new_y >= self.height:
            new_x, new_y = x, y
            reward = -100

        elif (new_x, new_y) in self.walls:
            new_x, new_y = x, y
            reward = -100

        elif (new_x, new_y) == self.finish:
            reward = 1000
            done = True

        next_state = (new_y * self.width) + new_x
        # print(current_state, next_state)

        return next_state, reward, done


State Space (ss) is defined as the size of the track. Action Space is 4; left, right, up, down.

In [257]:
class QTable:
    def __init__(self, ss, action_space):
        self.table = np.zeros((ss,action_space))

In [258]:
class Vehicle:
    def __init__(self, lr, df, ep, epo, ss, dim, action_names):
        self.learning_rate = lr
        self.discount_factor = df
        self.exploration_prob = ep
        self.epochs = epo
        self.dimensions = dim
        self.action_names = action_names
        self.qtable = QTable(ss, dim)

    def train(self, track):
        start_x, start_y = track.start
        start_index = (start_y * track.width) + start_x

        max_steps = 2000                                    # Prevent infinite loops
        
        for epoch in range(self.epochs): 
            current_state = start_index
            done = False
            steps = 0

            while not done and steps < max_steps:
                steps += 1
                    
                if np.random.rand() < self.exploration_prob:
                    action = np.random.randint(0, self.dimensions)          # Explore
                else:
                    action = np.argmax(self.qtable.table[current_state])    # Exploit

                next_state, reward, done = track.step(current_state, action)
                # print(f"State: {current_state}, Action: {action}, Next: {next_state}, Reward: {reward}")

                # Bellman equation for updating QTable
                
                self.qtable.table[current_state, action] += self.learning_rate * (reward + self.discount_factor * np.max(self.qtable.table[next_state]) - self.qtable.table[current_state, action])

                current_state = next_state

In [259]:
class Car(Vehicle):
    def __init__(self, lr, df, ep, epo, ss):
        actions = ['Left', 'Right', 'Up', 'Down']
        super().__init__(lr, df, ep, epo, ss, dim = 4, action_names = actions)

In [260]:
class Crawler(Vehicle):
    def __init__(self, lr, df, ep, epo, ss):
        actions = ['Left', 'Right', 'Up', 'Down', 'Left-Up', 'Right-Down', 'Left-Down', 'Right-Up']
        super().__init__(lr, df, ep, epo, ss, dim = 8, action_names = actions)

In [261]:
def main():

    # Read the track file, print track data, print track layout.

    my_track = RaceTrack()
    print(f"Track height: {my_track.height}, track width: {my_track.width}, start location: {my_track.start}, finish line {my_track.finish}\n")

    print("Visual presentation of the track:\n")
    for line in my_track.race_track_map:
        print(line)
    print()


    # Calculate the state space, ss.

    ss = (my_track.width * my_track.height)

    my_vehicle = Crawler(learning_rate, discount_factor, exploration_prob, epochs, ss)
    my_vehicle.train(my_track)

    
    df_qtable = pd.DataFrame(my_vehicle.qtable.table, columns = my_vehicle.action_names)
    df_qtable.index.name = 'State'

    pd.set_option('display.max_columns', None)  # Tillåter alla kolumner att visas
    pd.set_option('display.width', 1000)        # Ökar den tillåtna teckenbredden för utskriften

    print("Updated Q-table:")
    print(df_qtable.round(2))



In [262]:
if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: '/workspaces/LTU-Programming-Portfolio/d0045e_ht2026/race_track/auto2.txt'